In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms
import timm

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [3]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "tiny-vision-transformer",
    "learning_rate": 0.001,
    "epochs": 20,
    "pretrained":True
}

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ])
}


test_dir = "../BanglaLekha_8fold/test"

train_dataloaders = []
val_dataloaders = []

for i in range(1, 9):
    train_dir = f"../BanglaLekha_8fold/fold_{i}/train"
    val_dir = f"../BanglaLekha_8fold/fold_{i}/validation"
    train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
    val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
    train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)

    train_dataloaders.append(train_dataloader)
    val_dataloaders.append(val_dataloader)

test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [5]:

tiny_vit = timm.create_model("tiny_vit_11m_224", pretrained=True)

for params in tiny_vit.parameters():
    params.requires_grad = False

tiny_vit.reset_classifier(84)


total_params = sum(p.numel() for p in tiny_vit.parameters())

gpu = torch.device("cuda")
tiny_vit = tiny_vit.to(gpu)


In [6]:
print(tiny_vit)

TinyVit(
  (patch_embed): PatchEmbed(
    (conv1): ConvNorm(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (act): GELU(approximate='none')
    (conv2): ConvNorm(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (stages): Sequential(
    (0): ConvLayer(
      (blocks): Sequential(
        (0): MBConv(
          (conv1): ConvNorm(
            (conv): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (act1): GELU(approximate='none')
          (conv2): ConvNorm(
            (conv): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=256, bias=Fals

In [7]:
print(total_params)

10585688


In [8]:
import fine_tuning as ft

tiny_vit = ft.fine_tune(model=tiny_vit, model_name="tiny_vit", state="full") #Change the state for fine tuning 

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([
    {"params": tiny_vit.head.parameters(), "lr": 1e-3, "weight_decay": 1e-4},
    {"params": tiny_vit.stages.parameters(), "lr": 1e-5, "weight_decay": 1e-4},
    
])
epochs = model_config["epochs"]

In [9]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [11]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs, fold):
    
    max_val_accuracy = 0
    patience = 5
    count = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(tiny_vit, val_dataloader)

        if(val_accuracy > max_val_accuracy):
            max_val_accuracy = val_accuracy
            count = 0

            torch.save(model.state_dict(), f"saved_parameters/tiny_vit/tiny_vit_fold_{fold}.pth")
        else:
            count = count + 1

        if(count >= patience):
            break
        

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [14]:
for i in range(0, 8):
    print(f"Fold: {i+1}")
    train_model(tiny_vit, train_dataloaders[i], val_dataloaders[i], optimizer, criterion, model_config["epochs"], i+1)

Fold: 1
Epoch: 1, Training Loss: 0.24084526374760143, Validation Accuracy: 0.9668375037684654
Epoch: 2, Training Loss: 0.2150249921410477, Validation Accuracy: 0.9632197769068436
Epoch: 3, Training Loss: 0.200908645403833, Validation Accuracy: 0.962737413325294
Epoch: 4, Training Loss: 0.1884089768396856, Validation Accuracy: 0.9635815495930057
Epoch: 5, Training Loss: 0.17627578906105637, Validation Accuracy: 0.9638227313837805
Fold: 2
Epoch: 1, Training Loss: 0.17141212349597812, Validation Accuracy: 0.9753391618932771
Epoch: 2, Training Loss: 0.1597702033901137, Validation Accuracy: 0.9728670485378353
Epoch: 3, Training Loss: 0.1529382834965714, Validation Accuracy: 0.9726861621947543
Epoch: 4, Training Loss: 0.14780531267757943, Validation Accuracy: 0.9708170033162496
Epoch: 5, Training Loss: 0.13958086742760673, Validation Accuracy: 0.9693096171239072
Fold: 3
Epoch: 1, Training Loss: 0.1389719301700307, Validation Accuracy: 0.9813084112149533
Epoch: 2, Training Loss: 0.13046799788

In [15]:
max_accuracy = 0
for i in range(1, 9):
    tiny_vit.load_state_dict(torch.load(f"saved_parameters/tiny_vit/tiny_vit_fold_{i}.pth"))
    accuracy = validate_model(tiny_vit, test_dataloader)
    if(accuracy > max_accuracy):
        max_accuracy = accuracy
print(f"Accuracy: {100 * accuracy}")

Accuracy: 94.29965954626253
